In [15]:
import os
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

os.environ["DISPLAY"] = ":99"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-root"

options = Options()
options.binary_location = "/usr/bin/google-chrome"
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1366,768")
options.add_argument("--remote-debugging-port=9222")

driver = webdriver.Chrome(service=Service(), options=options)
driver.get("https://www.google.com")
print(driver.title)

Google


In [16]:
#Apertura del Navegador e Intervención Inicial
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

# --- CONFIGURACIÓN DEL ENTORNO VISUAL ---
options = Options()
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
# No usamos headless para que el estudiante vea la GUI en el contenedor
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

# Iniciar el driver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Navegar a Amazon
print(" Abriendo navegador en el contenedor...")
driver.get("https://www.amazon.com/s?k=laptop")

# --- PAUSA DE INTERVENCIÓN HUMANA ---
print("\n ACCIÓN REQUERIDA:")
print("1. Ve a la pestaña del escritorio del contenedor (VNC).")
print("2. Verifica si aparece un Captcha y resuélvelo manualmente.")
print("3. Una vez que veas el listado de laptops, regresa aquí.")

input("\n➔ Presiona ENTER en esta celda cuando estés listo para continuar...")

# --- VERIFICACIÓN DE BLOQUEO (Capítulo 7.4) ---
html_check = driver.page_source.lower()
if "captcha" in html_check or "robot" in html_check:
    print("ADVERTENCIA: Aún se detecta un bloqueo en el HTML.")
else:
    print("Confirmado: Navegador listo y sin bloqueos evidentes.")

 Abriendo navegador en el contenedor...

 ACCIÓN REQUERIDA:
1. Ve a la pestaña del escritorio del contenedor (VNC).
2. Verifica si aparece un Captcha y resuélvelo manualmente.
3. Una vez que veas el listado de laptops, regresa aquí.



➔ Presiona ENTER en esta celda cuando estés listo para continuar... 


ADVERTENCIA: Aún se detecta un bloqueo en el HTML.


In [17]:
# --- VERIFICACIÓN DE BLOQUEO (Capítulo 7.4) ---
if "robot" in driver.title.lower() or "captcha" in driver.title.lower():
    print(" Bloqueo real detectado")
else:
    print("Sin bloqueos evidentes")

Sin bloqueos evidentes


In [18]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['Mercado_Laboral_TI'] 
coleccion = db['AmazonLaptops'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [19]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print(" Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Mercado_Laboral_TI"
datos_finales = []   # Se define fuera del try para que siempre exista
driver = None        # Se define fuera del try para poder cerrarlo con seguridad

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"  # Ruta del binario de Chrome

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://www.amazon.es/s?k=laptops")

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")

        # Espera fija para que la página termine de cargar
        time.sleep(10)

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[data-component-type='s-search-result']"
        )

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.TAG_NAME, "h2").text
                precio = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
            except:
                # Si el bloque no tiene nombre o precio, se salta
                continue

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CLASS_NAME, "s-pagination-next")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f" Extracción terminada: {len(datos_finales)} productos.")

except Exception as e:
    print(f" Error en Selenium: {e}")

finally:
    # Cierra el navegador solo si logró abrirse
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    #client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["Mercado_Laboral_TI"]
    coleccion = db["AmazonLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpia el valor antes de convertirlo
            v_limpio = str(d["valor"]).replace(".", "").replace(",", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos cargados en MongoDB correctamente.")
    else:
        print(" No hay datos para guardar.")

except Exception as e:
    print(f" Error en MongoDB: {e}")
    

 Limpieza de procesos y temporales completada.
 Navegador iniciado correctamente.
--- Procesando Página 1 ---
--- Procesando Página 2 ---
--- Procesando Página 3 ---
 Extracción terminada: 129 productos.
 Datos cargados en MongoDB correctamente.


In [24]:
from pyspark.sql import SparkSession

# Forzamos cierre de sesiones previas con versiones erróneas
try: spark.stop()
except: pass

spark = SparkSession.builder \
    .appName("Analisis_BigData_Final") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/Mercado_Laboral_TI.AmazonLaptops") \
    .getOrCreate()

df = spark.read.format("mongodb").load()
print(f"Éxito total: {df.count()} registros.")
df.show(5)

Éxito total: 389 registros.
+--------------------+-------------------+------------------+--------------------+-----+
|                 _id|      fecha_captura|             grupo|       identificador|valor|
+--------------------+-------------------+------------------+--------------------+-----+
|69e2918faf79f3331...|2026-04-17 20:00:36|Mercado_Laboral_TI|15.6" FHD Laptop ...|399.0|
|69e2918faf79f3331...|2026-04-17 20:00:37|Mercado_Laboral_TI|Laptop 15.6 Inch ...|349.0|
|69e2918faf79f3331...|2026-04-17 20:00:37|Mercado_Laboral_TI|ACEMAGIC AX18 Lap...|469.0|
|69e2918faf79f3331...|2026-04-17 20:00:37|Mercado_Laboral_TI|Laptop Ultralight...|369.0|
|69e2918faf79f3331...|2026-04-17 20:00:37|Mercado_Laboral_TI|15.6 Inch Laptop,...|289.0|
+--------------------+-------------------+------------------+--------------------+-----+
only showing top 5 rows



In [25]:
from pyspark.sql.functions import col, split, when, avg
# 2. Transformación de Negocio: Extraer la MARCA
# En Amazon, la primera palabra del título suele ser la marca.
df_transformado = df.withColumn("marca", split(col("identificador"), " ")[0])

# 3. Limpieza de Outliers: Filtrar laptops con precios erróneos (ej: < 100 euros)
df_validado = df_transformado.filter(col("valor") > 100)

# 4. Agregación: Precio promedio por Marca
reporte_marcas = df_validado.groupBy("marca").agg(
    avg("valor").alias("precio_promedio")
).orderBy(col("precio_promedio").desc())

reporte_marcas.show()

+---------+------------------+
|    marca|   precio_promedio|
+---------+------------------+
|      MSI|            1558.0|
|       LG|          1337.875|
|   Lenovo| 853.2162162162163|
|     Acer| 848.3076923076923|
|     ASUS| 841.6363636363636|
|Microsoft|             770.0|
|     DELL|            764.25|
| ACEMAGIC| 502.8095238095238|
|  Lenovo,|             492.5|
|    Ryzen|             487.0|
|     acer|            461.75|
|       16|             439.0|
|        2|             423.0|
|      GMR|            395.25|
|   Laptop|395.06451612903226|
|   NGTeco|             389.0|
|    15.6"|             374.0|
|       HP|            354.92|
|     2021|             353.0|
|     15.6| 338.5238095238095|
+---------+------------------+
only showing top 20 rows

